# Lab 05 — Alert Reduction

把多個低階 anomaly flags 聚合為更少、更有語意的 alerts，降低告警疲勞。

In [ ]:
from pathlib import Path
from IPython.display import SVG, display

def find_project_root(start=Path.cwd()):
    start = start.resolve()
    for base in (start, *start.parents):
        if (base / "environment.yml").exists():
            return base
    raise FileNotFoundError("Could not find project root containing environment.yml")

svg_candidates = [
    Path("diagrams/lab05_pipeline_position.svg"),
    find_project_root() / "labs/self-study" / "diagrams/lab05_pipeline_position.svg",
]
for svg_path in svg_candidates:
    if svg_path.exists():
        display(SVG(filename=str(svg_path)))
        break
else:
    raise FileNotFoundError("Could not find diagram: diagrams/lab05_pipeline_position.svg")


## Lab 05 概覽

### 學習目標

1. 理解 alert fatigue（告警疲勞）如何讓監控系統失效
2. 實作 time-視窗 grouping 把重複 flag 合併成 alert 事件
3. 設計問題分類規則，為 alert 加上語意標籤
4. 量化降噪效果，評估是否遺漏真實事件

### 前置條件

Lab 02 完成（`outputs/self-study/baseline_anomaly_flags.csv` 存在）

## 設計主線：把 flags 轉成工程師能處理的事件

本章的實務連結是告警疲勞。偵測層輸出的每個 flag 都可能是症狀，但工程師需要處理的是事件，而不是成千上萬個二元標記。

設計告警降噪時請問三個問題：

1. **哪些 flags 屬於同一事件**：同一 port、同一 metric、相近時間是否應該合併？跨 port 同時異常是否代表同一根因？
2. **grouping window 多長**：太短會碎裂成多個 alerts，太長會把不同事件混在一起。
3. **分類是否能導向行動**：problem type 應該能對應到 runbook、升級路徑或 RCA context。

設計原則：降噪不是把告警變少而已。好的降噪要保留處置所需的證據。


In [ ]:
from pathlib import Path
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 160)

def show_fig(fig):
    display(fig)
    plt.close(fig)

PROJECT_ROOT = Path.cwd()
while PROJECT_ROOT != PROJECT_ROOT.parent:
    if (PROJECT_ROOT / "environment.yml").exists():
        break
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_SYNTHETIC = PROJECT_ROOT / "data" / "synthetic"
DATA_PROCESSED = PROJECT_ROOT / "outputs" / "self-study"
REPORTS = PROJECT_ROOT / "reports"
DATA_PROCESSED.mkdir(parents=True, exist_ok=True)
REPORTS.mkdir(parents=True, exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")

In [ ]:
flags = pd.read_csv(DATA_PROCESSED / "baseline_anomaly_flags.csv", parse_dates=["timestamp"])
display(flags.head())

---
📖 **概念說明 ▸ 講師引導** — 講師帶領說明約 5 分鐘，請先不要執行 cell。

---

## 📖 概念：Alert Fatigue — 告警太多，等於沒有告警

### 問題的本質

Lab 02 和 Lab 03 的輸出是一個 binary flag：每個時間點每個 metric 都有 0 或 1。
如果你有 20 個 port × 5 個 metric × 每 5 分鐘一個點：

```
20 ports × 5 metrics × 288 points/day = 28,800 possible flags/day
```

即使 flag rate 只有 5%，也是 **1,440 則告警 / 天**——等於每分鐘一則。

**這是真實 NOC（Network Operations Center）的頭號問題。**
研究顯示，當告警量超過運維工程師能處理的上限時：
- 工程師開始忽略告警（alert desensitization）
- 重要事件在噪音中被淹沒
- 疲勞導致真實事件的平均回應時間延長

### Alert Reduction 的三個層次

| 層次 | 技術 | 效果 |
|---|---|---|
| **去重（Deduplication）** | 同一 port 同一 metric 連續的 flag 合併成一個 alert | 把 N 個連續 flag 變成 1 個事件 |
| **分組（Grouping）** | 同一時間多個 port / metric 的 alert 合併 | 把同一根因的多個症狀合成一個事件 |
| **抑制（Suppression）** | 已知的計畫維護、正常高峰不告警 | 去除「已解釋的」告警 |

本 lab 實作前兩個層次。

### 降噪的風險

降噪的最大風險：把真實事件的告警也合併掉了，導致漏報。
這就是為什麼需要人類驗證：**自動化的降噪規則必須有人確認不會遺漏關鍵事件**。

## Step 1 - 將 anomaly flags 展開成 raw alerts

In [ ]:
flag_cols = [c for c in flags.columns if "__flag_" in c]
print(f"flag columns: {len(flag_cols)}")

raw_rows = []
for _, row in flags.iterrows():
    for flag in flag_cols:
        if bool(row[flag]):
            metric = flag.split("__flag_", 1)[0]
            raw_rows.append(
                {
                    "timestamp": row["timestamp"],
                    "device_id": row["device_id"],
                    "port_id": row["port_id"],
                    "event_label": row["event_label"],
                    "event_id": row["event_id"],
                    "metric": metric,
                    "alert_name": flag,
                }
            )
raw_alerts = pd.DataFrame(raw_rows)
print(f"raw alerts: {len(raw_alerts):,}")
display(raw_alerts.head(10))


---
💻 **自行執行** — 閱讀說明並依序執行以下 cell。有疑問請舉手。

---

## Step 2 - 視覺化：Raw alert volume

這對應 Grafana 的 raw alert count panel。

In [ ]:
alert_counts = raw_alerts.set_index("timestamp").resample("1h").size()
fig, ax = plt.subplots(figsize=(14, 4))
alert_counts.plot(ax=ax, color="tab:red", linewidth=1)
ax.set_title("Raw alert count per hour")
ax.set_ylabel("raw alerts")
ax.grid(alpha=0.25)
plt.tight_layout()
show_fig(fig)

---
📖 **概念說明 ▸ 講師引導** — 講師帶領說明約 5 分鐘，請先不要執行 cell。

---

## 📖 概念：Alert Grouping 與問題分類

### 時間視窗 分組

時間視窗 grouping 的核心邏輯：
「在時間窗口 W 內，由相同類型的 metric 觸發的告警，屬於同一個事件。」

```
Raw flags (every 5 minutes):
  14:00 port-7427 error_rate_flag=1
  14:05 port-7427 error_rate_flag=1
  14:10 port-7427 error_rate_flag=1
  14:15 port-7427 discard_rate_flag=1
  14:00 port-7428 error_rate_flag=1

After 15-minute window →
  Alert #1: 14:00–14:10, port-7427, error+discard, severity=High
  Alert #2: 14:00, port-7428, error_rate
```

### 分組 vs 相關性分析

本課程使用的是簡單的 **time-視窗 grouping**（按時間窗口合併）。
更複雜的做法是 **correlation-based grouping**：先計算 alert 之間的相關性（拓樸距離、時間重疊、共同根因），再用圖聚類算法分組。

| | 時間視窗 grouping | Correlation-based grouping |
|---|---|---|
| 複雜度 | 低 | 高（需要拓樸資訊 + 聚類算法） |
| 可解釋性 | 高（規則清晰） | 低（黑盒聚類） |
| 效果 | 對簡單場景夠用 | 對複雜多設備事件更準確 |
| 適合場景 | 入門、資料量有限 | 企業級 NOC、有 CMDB 的環境 |

本課程選 time-視窗 grouping 是因為它透明、可審計、容易調整。

## Step 3 - 設計問題分類與 15 分鐘聚合規則

In [ ]:
def classify_problem(metrics: set[str], affected_ports: int) -> str:
    if "broadcast" in metrics and affected_ports > 1:
        return "Broadcast storm / L2 loop"
    if "multicast" in metrics and affected_ports > 1:
        return "Multicast flooding"
    if {"traffic", "discards"}.issubset(metrics) and "errors" not in metrics:
        return "Queue congestion"
    if "discards" in metrics:
        return "Queue / buffer pressure"
    if "errors" in metrics and "traffic" not in metrics:
        return "Link quality issue"
    if "unknown_protocol" in metrics:
        return "Unknown protocol / scan"
    if "packets" in metrics and "traffic" in metrics:
        return "Traffic or packet surge"
    if "packets" in metrics:
        return "Packet surge / possible scan"
    if "traffic" in metrics:
        return "Traffic surge / capacity pressure"
    if "broadcast" in metrics:
        return "Broadcast anomaly"
    if "multicast" in metrics:
        return "Multicast anomaly"
    return "General anomaly"


raw_alerts = raw_alerts.sort_values(["port_id", "timestamp"])
raw_alerts["bucket"] = raw_alerts["timestamp"].dt.floor("15min")
reduced_rows = []
for (bucket, alert_name), g in raw_alerts.groupby(["bucket", "alert_name"]):
    metrics = set(g["metric"])
    affected_ports = g["port_id"].nunique()
    severity = min(100, 20 + 8 * len(g) + 15 * affected_ports + 12 * len(metrics))
    reduced_rows.append(
        {
            "start_time": g["timestamp"].min(),
            "end_time": g["timestamp"].max(),
            "problem_type": classify_problem(metrics, affected_ports),
            "affected_ports": ",".join(sorted(g["port_id"].unique())),
            "affected_port_count": affected_ports,
            "evidence_metrics": ",".join(sorted(metrics)),
            "raw_alert_count": len(g),
            "severity_score": severity,
            "event_labels": ",".join(sorted(set(g["event_label"]))),
        }
    )
reduced = pd.DataFrame(reduced_rows).sort_values(["start_time", "severity_score"], ascending=[True, False])
display(reduced.head(12))

## Step 4 - 視覺化：Raw vs Reduced、severity distribution、Top affected ports

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
pd.Series({"raw_alerts": len(raw_alerts), "reduced_alerts": len(reduced)}).plot(kind="bar", ax=axes[0], color=["tab:red", "tab:blue"])
axes[0].set_title("Raw vs Reduced")
axes[0].set_ylabel("count")

reduced["severity_score"].plot(kind="hist", bins=20, ax=axes[1], color="tab:orange")
axes[1].set_title("Severity distribution")

top_ports = raw_alerts["port_id"].value_counts().head(8)
top_ports.sort_values().plot(kind="barh", ax=axes[2], color="tab:green")
axes[2].set_title("Top affected ports")
plt.tight_layout()
show_fig(fig)

---

## ⚖️ 方法比較：時間視窗 Grouping vs Correlation-based Grouping

| | 時間視窗 Grouping | Correlation-based Grouping |
|---|---|---|
| **需要什麼資料** | 時間戳記 + metric 類型 | 時間戳記 + 拓樸鄰接表 + 歷史相關性 |
| **可調整性** | 直接調整窗口大小（分鐘）| 需要調整距離閾值 + 聚類參數 |
| **漏報風險** | 中（同一事件跨越多個窗口可能被切成兩個 alert）| 低（可以跨更長Time整合） |
| **誤合風險** | 低（只合併同一時間段的）| 中（可能把兩個不同事件誤認為同一根因）|
| **可解釋性** | 高——工程師可以看懂規則 | 中——需要理解聚類算法 |

### 本課程的建議

時間視窗 grouping 是一個**可以立即部署、透明可稽核**的起點。
它的主要弱點是對「跨越長Time的漸進式事件」不友善——
例如一個記憶體洩漏用了 4 個小時才觸發第一個 alert，
這時 15 分鐘的窗口無法有效分組。

升級路徑：在 time-視窗 grouping 後，用事件的 `problem_type` 作為第二層合併依據，
可以把「同一天同一問題類型的多個小事件」進一步聚合。

---
🔧 **探索練習** — 修改指定參數，觀察結果變化。

---

## 🔧 探索練習：分組 視窗 大小

在 Step 3 的 code cell 中找到Time分組窗口參數（通常是 `"15min"` 或類似），
改成 `"30min"`，重新執行 Step 4 的視覺化。

觀察：
- Reduced alert 數量是否進一步下降？
- Alert timeline（Step 5）的事件是否有任何合併到你認為不應合併的情況？
- 降噪比例提升了，但「保真度」下降了多少？

---

> 「如果你把窗口設為 1 小時，理論上可以把很多小 alert 合成一個大 alert。為什麼你不應該這樣做？」

> 「在視覺化圖裡，有哪個時段的 raw alert 很多但 reduced alert 很少？那段Time實際發生了什麼？」

> 「降噪之後，如何驗證你沒有漏掉任何真實事件？你需要什麼資料才能做這個驗證？」

---
💻 **自行執行** — 閱讀說明並依序執行以下 cell。有疑問請舉手。

---

## Step 5 - Alert timeline

In [ ]:
timeline = reduced.copy()
timeline["mid_time"] = timeline["start_time"] + (timeline["end_time"] - timeline["start_time"]) / 2
types = {name: i for i, name in enumerate(sorted(timeline["problem_type"].unique()))}
timeline["y"] = timeline["problem_type"].map(types)

fig, ax = plt.subplots(figsize=(14, 6))
scatter = ax.scatter(timeline["mid_time"], timeline["y"], s=timeline["severity_score"] * 2, c=timeline["severity_score"], cmap="viridis", alpha=0.75)
ax.set_yticks(list(types.values()))
ax.set_yticklabels(list(types.keys()))
ax.set_title("Reduced alert timeline")
ax.grid(alpha=0.25)
plt.colorbar(scatter, ax=ax, label="severity")
plt.tight_layout()
show_fig(fig)

In [ ]:
raw_path = DATA_PROCESSED / "raw_alerts.csv"
reduced_path = DATA_PROCESSED / "reduced_alerts.csv"
raw_alerts.to_csv(raw_path, index=False)
reduced.to_csv(reduced_path, index=False)
print(f"saved: {raw_path}")
print(f"saved: {reduced_path}")

## ⚠ 人類驗證點 #5 — 降噪窗口的大小是時效性 vs 簡潔性的取捨

告警降噪不是「做到告警最少就好」——它是在「告警太多讓人麻痺」和「告警太少讓事件被遺漏」之間找平衡點。

### 如何判斷

降噪後，評估以下三個指標：

1. **降噪比**：raw alerts / reduced alerts。比值太低（< 5x）表示降噪效果有限；太高（> 50x）可能在合併不相關的事件
2. **事件完整性**：events.csv 裡的每個已知事件，在 reduced alerts 裡都有對應記錄嗎？
3. **平均 alert 持續時間**：合理的事件持續時間是分鐘到小時，不是幾秒或幾天

### 讓你重新考慮的信號

- 一個 reduced alert 的 `affected_port_count` > 10 → 可能把多個不相關事件合併了，窗口過大
- 已知的 `broadcast_storm` 事件和 `traffic_surge` 事件被合併成同一個 alert → 分類規則需要調整
- `duration_minutes` 中位數 < 5 分鐘 → 窗口可能太小，可考慮加大

### 🔧 探索練習

在 Step 3 的 code cell 找到以下參數：

```python
WINDOW_MINUTES = 15    # change to 5 or 30, observe how alert count and affected_port_count shift
```

> 「如果你是 NOC 工程師，你希望一個班次（8 小時）裡看到幾個 alert？10 個？100 個？」

> 「目前的 `severity_score` 公式是什麼？它是否反映了真正的業務衝擊？」

> 「你的組織目前的告警是否有做降噪？如果有，用的是什麼方式？」
